In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor


In [2]:
path = Path('.')
train = pd.read_csv(path / 'period_1_train_data.csv')
test = pd.read_csv(path / 'test_x.csv')


In [3]:
train = train.drop_duplicates().reset_index(drop=True)
train = train[(train['price_target'] > 0) & (train['square'] > 10)]
train = train[(train['floor'] >= 1) & (train['floor'] <= 50)].reset_index(drop=True)
test['floor'] = test['floor'].clip(1, 50)

train = train.replace(-999, np.nan)
test = test.replace(-999, np.nan)

bad_cols = ['location_depth', 'location_depth.1', 'location_depth.2']
train = train.drop(columns=[c for c in bad_cols if c in train.columns])
test = test.drop(columns=[c for c in bad_cols if c in test.columns])


In [4]:
def fix_rooms(x):
    if x == 'студия':
        return 0
    if x == '>=4':
        return 4
    return pd.to_numeric(x, errors='coerce')

for df in [train, test]:
    df['rooms'] = df['rooms_4'].apply(fix_rooms).fillna(1)
    df['rooms_sq'] = df['rooms'] ** 2
    df['agreement_date'] = pd.to_datetime(df['agreement_date'])

min_date = train['agreement_date'].min()
for df in [train, test]:
    df['year'] = df['agreement_date'].dt.year
    df['month'] = df['agreement_date'].dt.month
    df['months_from_start'] = (df['agreement_date'] - min_date).dt.days / 30
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)


In [5]:
cat_cols = ['hc_name_cat', 'developer_cat', 'district_cat', 'region_name_cat', 'class_cat', 'stage_cat', 'interior_cat']
cat_cols = [c for c in cat_cols if c in train.columns]

for col in cat_cols:
    train[col] = train[col].astype(str).str.replace('.0$', '', regex=True).fillna('missing')
    test[col] = test[col].astype(str).str.replace('.0$', '', regex=True).fillna('missing')


In [6]:
def add_base(train, test):
    base = train['price_target'].mean()
    maps = [
        train.groupby('hc_name_cat')['price_target'].mean(),
        train.groupby('developer_cat')['price_target'].mean(),
        train.groupby('district_cat')['price_target'].mean(),
    ]
    for df in [train, test]:
        df['base'] = np.nan
        for col, mp in zip(['hc_name_cat', 'developer_cat', 'district_cat'], maps):
            if col in df.columns:
                df['base'] = df['base'].fillna(df[col].map(mp))
        df['base'] = df['base'].fillna(base)
        df['base_log'] = np.log(df['base'])
    return train, test

train, test = add_base(train, test)
train['target_log_residual'] = np.log(train['price_target']) - train['base_log']


In [7]:
def target_encode(train, test, cols, target='price_target', n_splits=5, smooth=50):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    global_mean = train[target].mean()

    for col in cols:
        train[f'{col}_te'] = np.nan
        train[f'{col}_cnt'] = 0

        for train_idx, val_idx in kf.split(train):
            part = train.iloc[train_idx]
            stats = part.groupby(col)[target].agg(['mean', 'count'])
            enc = (stats['mean'] * stats['count'] + global_mean * smooth) / (stats['count'] + smooth)
            idx = train.index[val_idx]
            train.loc[idx, f'{col}_te'] = train.loc[idx, col].map(enc)
            train.loc[idx, f'{col}_cnt'] = train.loc[idx, col].map(stats['count'])

        stats = train.groupby(col)[target].agg(['mean', 'count'])
        enc = (stats['mean'] * stats['count'] + global_mean * smooth) / (stats['count'] + smooth)
        train[f'{col}_te'] = train[f'{col}_te'].fillna(global_mean)
        train[f'{col}_cnt'] = train[f'{col}_cnt'].fillna(0)
        test[f'{col}_te'] = test[col].map(enc).fillna(global_mean)
        test[f'{col}_cnt'] = test[col].map(stats['count']).fillna(0)

    return train, test

te_cols = [c for c in ['hc_name_cat', 'developer_cat', 'district_cat'] if c in train.columns]
train, test = target_encode(train, test, te_cols)


In [8]:
def add_group_stats(train, test, col):
    stats = train.groupby(col).agg(
        square_mean=('square', 'mean'),
        square_std=('square', 'std'),
        floors_mean=('location_max_levels_max', 'mean'),
        density_mean=('location_mean_area_density', 'mean')
    )
    for name in stats.columns:
        train[f'{col}_{name}'] = train[col].map(stats[name])
        test[f'{col}_{name}'] = test[col].map(stats[name])
    return train, test

for col in [c for c in ['developer_cat', 'district_cat'] if c in train.columns]:
    train, test = add_group_stats(train, test, col)


In [9]:
for df in [train, test]:
    df['log_square'] = np.log1p(df['square'])
    df['floor_rel'] = df['floor'] / (df['location_max_levels_max'] + 1)
    df['is_first_floor'] = (df['floor'] == 1).astype(int)
    df['is_last_floor'] = (df['floor'] == df['location_max_levels_max']).astype(int)
    df['square_x_floor'] = df['square'] * df['floor_rel']
    df['transport_density'] = df['location_public_transport_stop_position_cnt'] / (df['location_mean_area_density'] + 1)
    df['commercial_density'] = df['location_commercial_cnt'] / (df['location_mean_area_density'] + 1)

class_order = {'2581': 3, '27353': 2, '97865': 1}
stage_order = {'103': 1, '2665': 2, '2700': 3, '3321': 4, '7983': 5, '12638': 6, '27728': 7, '70661': 8}

for df in [train, test]:
    df['class_rank'] = df['class_cat'].map(class_order).fillna(2)
    df['stage_rank'] = df['stage_cat'].map(stage_order).fillna(4)
    df['square_x_class'] = df['square'] * df['class_rank']
    df['rooms_x_class'] = df['rooms'] * df['class_rank']


In [10]:
missing_cols = [c for c in train.columns if c in test.columns and train[c].isna().any()]
missing_cols = [c for c in missing_cols if pd.api.types.is_numeric_dtype(train[c])]

train_flags = {f'{c}_isna': train[c].isna().astype(int) for c in missing_cols}
test_flags = {f'{c}_isna': test[c].isna().astype(int) for c in missing_cols}

train = pd.concat([train, pd.DataFrame(train_flags, index=train.index)], axis=1).copy()
test = pd.concat([test, pd.DataFrame(test_flags, index=test.index)], axis=1).copy()


In [11]:
num_cols = train.select_dtypes(include=np.number).columns.tolist()
drop_cols = ['price_target', 'target_log_residual']
features = [c for c in num_cols if c not in drop_cols]
features = [c for c in features if c in test.columns]

model_cat_cols = ['region_name_cat', 'class_cat', 'stage_cat', 'interior_cat']
model_cat_cols = [c for c in model_cat_cols if c in train.columns]
features = features + model_cat_cols

for col in features:
    if col in model_cat_cols:
        train[col] = train[col].fillna('missing').astype(str)
        test[col] = test[col].fillna('missing').astype(str)
    else:
        value = train[col].median()
        train[col] = train[col].fillna(value)
        test[col] = test[col].fillna(value)


In [12]:
def make_lgb_data(train_df, other_df, features, cat_cols):
    num_features = [c for c in features if c not in cat_cols]
    left = train_df[num_features].copy()
    right = other_df[num_features].copy()

    for col in cat_cols:
        values = pd.concat([train_df[col], other_df[col]]).astype(str).unique()
        mapping = {v: i for i, v in enumerate(values)}
        left[col] = train_df[col].astype(str).map(mapping).fillna(-1)
        right[col] = other_df[col].astype(str).map(mapping).fillna(-1)

    return left, right

train_part, val_part = train_test_split(train, test_size=0.2, random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cb_oof = np.zeros(len(train_part))
cb_val = np.zeros(len(val_part))

for tr_idx, hold_idx in kf.split(train_part):
    tr = train_part.iloc[tr_idx]
    hold = train_part.iloc[hold_idx]

    cb = CatBoostRegressor(
        iterations=2000,
        learning_rate=0.02,
        depth=8,
        l2_leaf_reg=6,
        loss_function='MAE',
        random_seed=42,
        verbose=False,
        allow_writing_files=False
    )

    cb.fit(tr[features], tr['target_log_residual'], cat_features=model_cat_cols)
    cb_oof[hold_idx] = cb.predict(hold[features])
    cb_val += cb.predict(val_part[features]) / kf.n_splits

X_train_lgb, X_val_lgb = make_lgb_data(train_part, val_part, features, model_cat_cols)
X_train_lgb['cb_pred'] = cb_oof
X_val_lgb['cb_pred'] = cb_val

lgb = LGBMRegressor(
    n_estimators=3000,
    learning_rate=0.015,
    max_depth=8,
    num_leaves=96,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.2,
    reg_lambda=0.2,
    min_child_samples=20,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgb.fit(X_train_lgb, train_part['target_log_residual'])
val_residual = lgb.predict(X_val_lgb)
val_pred = np.exp(val_part['base_log'] + val_residual)
mean_absolute_error(val_part['price_target'], val_pred)


612.742179748117

In [13]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cb_oof = np.zeros(len(train))
cb_test = np.zeros(len(test))

for tr_idx, hold_idx in kf.split(train):
    tr = train.iloc[tr_idx]
    hold = train.iloc[hold_idx]

    cb = CatBoostRegressor(
        iterations=2000,
        learning_rate=0.02,
        depth=8,
        l2_leaf_reg=6,
        loss_function='MAE',
        random_seed=42,
        verbose=False,
        allow_writing_files=False
    )

    cb.fit(tr[features], tr['target_log_residual'], cat_features=model_cat_cols)
    cb_oof[hold_idx] = cb.predict(hold[features])
    cb_test += cb.predict(test[features]) / kf.n_splits

X_train_lgb, X_test_lgb = make_lgb_data(train, test, features, model_cat_cols)
X_train_lgb['cb_pred'] = cb_oof
X_test_lgb['cb_pred'] = cb_test

lgb = LGBMRegressor(
    n_estimators=3000,
    learning_rate=0.015,
    max_depth=8,
    num_leaves=96,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.2,
    reg_lambda=0.2,
    min_child_samples=20,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgb.fit(X_train_lgb, train['target_log_residual'])
preds = np.exp(test['base_log'] + lgb.predict(X_test_lgb))
preds = np.maximum(preds, 10000)


In [14]:
submission = pd.DataFrame({'id': test.index, 'price_target': preds})
submission.to_csv('submission_5.csv', index=False)
